# EPCAPE data availability: % of each month with data

The *DataAvailability* tab of `Table-EPCAPE_SeasonalAverages.xlsx` gives, for each instrument, the fraction of each campaign month (Feb 2023 – Feb 2024) with data. Most instruments are already filled in; this notebook computes the same thing for the ones that are not:

* **Part 1 (Sections 1–5):** the Fog Droplet Monitor (DMT FM-120) at Mt. Soledad (sheet row 50).
* **Part 2 (Sections 6–8):** the JPL Ka-, W- and G-band radars at the Scripps Pier (Matt Lebsock; sheet row 49).

**Data.** `data/processed/FM120/Monthly FM-120 Files/` holds one CSV per month (`YYYY-MM.csv`) of **5-minute averages**. Each row is one 5-min sample; times with no data simply have no row.

**Method: total time with data.** Each valid 5-min sample stands for 5 minutes of measurement, so

$$\text{availability}_{\text{month}} = \frac{N_{\text{samples}} \times 5\ \text{min}}{\text{total time in the month}}$$

For comparison, the notebook also gives the fraction of *days* with at least one sample. That number is higher, because a day with a single 5-min sample counts as a full day.

**Two denominators.** EPCAPE ran 2023-02-15 to 2024-02-14 (`config.yaml`, `campaign:`), so the first and last months are only half inside the campaign. Two tables are computed, and only the denominator differs between them:

* **Campaign days only**: Feb 2023 counts Feb 15–28 and Feb 2024 counts Feb 1–14. The ARM rows of the tab show 100 % for Feb 2023, so they appear to be computed this way.
* **Whole month**: every month counts all of its hours (Feb 2023 = 28 days, Feb 2024 = 29 days). This is what Lynn asked for (2026-10-15).

The numerator also differs in one place. Table A counts only samples inside the campaign window. Table B counts every sample recorded in the month, which adds the 17.1 h recorded on 2024-02-15 after the campaign ended (Lynn's choice, 2026-10-15). All other months are identical in the two tables.

In [1]:
# --- Imports ------------------------------------------------------------------
from pathlib import Path

import pandas as pd
import yaml

In [2]:
# --- Settings -----------------------------------------------------------------
# The notebook lives in EPCAPE/comparisons/data_availability/, so the EPCAPE folder is two levels up.
EPCAPE_DIR = Path.cwd().parents[1]
FM120_DIR = EPCAPE_DIR / "data" / "processed" / "FM120" / "Monthly FM-120 Files"

# Length of one FM-120 averaging interval: each row represents 5 minutes of measurement.
SAMPLE_INTERVAL = pd.Timedelta(minutes=5)

# Campaign window, read from config.yaml so it matches the rest of the EPCAPE code.
with open(EPCAPE_DIR / "config.yaml") as f:
    campaign = yaml.safe_load(f)["campaign"]
CAMPAIGN_START = pd.Timestamp(campaign["start"])                       # 2023-02-15 00:00 (inclusive)
CAMPAIGN_END = pd.Timestamp(campaign["end"]) + pd.Timedelta(days=1)   # 2024-02-15 00:00 (exclusive)

# Whole calendar months spanning the campaign: 2023-02-01 00:00 to 2024-03-01 00:00 (exclusive).
FULL_MONTHS_START = CAMPAIGN_START.to_period("M").start_time
FULL_MONTHS_END = (CAMPAIGN_END - pd.Timedelta(days=1)).to_period("M").end_time.ceil("D")

print("FM-120 folder:", FM120_DIR)
print("Campaign:", CAMPAIGN_START, "to", CAMPAIGN_END, "(end exclusive)")

FM-120 folder: /Users/andrewbuggee/Documents/VS_CODE/Python-Research/EPCAPE/data/processed/FM120/Monthly FM-120 Files
Campaign: 2023-02-15 00:00:00 to 2024-02-15 00:00:00 (end exclusive)


# Part 1: FM-120

## 1. Read the monthly 5-min files into one time series

* The first column of each CSV is the timestamp; all months are stacked into one table indexed by time.
* A sample counts as **valid** if any of the bulk quantities (temperature, airspeed, number concentration, LWC, MVD, ED) is present. In these files every row has all of them, so this only guards against future gaps.
* `fm120_all` keeps every sample (table B). `fm120` keeps only samples inside the campaign window (table A, and the gap analysis). The Feb 2024 file runs to 2024-02-15 17:00.
* Times are UTC, and each timestamp marks the start of its 5-min interval (`data/processed/readme_FM120.md`: files run 00:00 to 23:55 UTC).

In [3]:
# --- Load and stack all monthly files -----------------------------------------
files = sorted(FM120_DIR.glob("*.csv"))
fm120 = pd.concat([pd.read_csv(f, index_col=0, parse_dates=True) for f in files]).sort_index()

# Each 5-min interval should appear once and sit on the 5-min grid; stop if not,
# since the "N_samples x 5 min" method below relies on both.
assert not fm120.index.duplicated().any(), "duplicate timestamps"
assert (fm120.index == fm120.index.floor(SAMPLE_INTERVAL)).all(), "timestamps off the 5-min grid"

# Keep samples where at least one bulk quantity was measured.
BULK_COLUMNS = ["T Ambient (C)", "PAS (m/s)", "Number Conc (#/cm^3)", "LWC (g/m^3)", "MVD (um)", "ED (um)"]
fm120 = fm120.dropna(subset=BULK_COLUMNS, how="all")

# Keep every sample for the whole-month table, and the campaign-window subset for everything else.
fm120_all = fm120
in_campaign = (fm120.index >= CAMPAIGN_START) & (fm120.index < CAMPAIGN_END)
print(f"Samples outside the campaign window (only used in table B): {(~in_campaign).sum()}")
fm120 = fm120[in_campaign]

print(f"{len(files)} files, {len(fm120)} valid 5-min samples, "
      f"{fm120.index.min()} to {fm120.index.max()}")

Samples outside the campaign window (only used in table B): 205
13 files, 89906 valid 5-min samples, 2023-02-16 16:40:00 to 2024-02-14 23:55:00


## 2. Time with data vs. total time in each month

`availability()` builds the table for one window, the campaign window or whole calendar months. It counts the samples recorded inside that window.

In [4]:
def availability(data: pd.DataFrame, window_start: pd.Timestamp,
                 window_end: pd.Timestamp) -> pd.DataFrame:
    """Monthly availability over the window window_start (inclusive) to window_end
    (exclusive), counting the 5-min samples of `data` that fall inside it."""
    # --- Numerator: time with data in each month -------------------------------
    data = data[(data.index >= window_start) & (data.index < window_end)]
    month = data.index.to_period("M")
    hours_with_data = data.groupby(month).size() * SAMPLE_INTERVAL / pd.Timedelta(hours=1)
    # Days with at least one sample, for comparison with the time-based number.
    days_with_data = pd.Series(data.index.normalize().unique()).dt.to_period("M").value_counts()

    # --- Denominator: every day in the window, counted per month ---------------
    all_days = pd.date_range(window_start, window_end, freq="D", inclusive="left")
    possible_days = pd.Series(1, index=all_days).groupby(all_days.to_period("M")).size()

    # --- Combine ----------------------------------------------------------------
    table = pd.DataFrame({
        "possible_hours": possible_days * 24,
        "hours_with_data": hours_with_data,
        "possible_days": possible_days,
        "days_with_data": days_with_data,
    }).fillna(0)
    table["time_fraction"] = table["hours_with_data"] / table["possible_hours"]
    table["day_fraction"] = table["days_with_data"] / table["possible_days"]
    table.index.name = "month"
    return table


# Table A: Feb 2023 and Feb 2024 count only their campaign days.
avail = availability(fm120, CAMPAIGN_START, CAMPAIGN_END)
# Table B: every month counts all of its hours, and all data recorded in it.
avail_full = availability(fm120_all, FULL_MONTHS_START, FULL_MONTHS_END)

print("A. Campaign days only")
display(avail.round(3))
print("B. Whole month")
display(avail_full.round(3))

A. Campaign days only


,possible_hours,hours_with_data,possible_days,days_with_data,time_fraction,day_fraction
month,,,,,,
2023-02,336,81.917,14,9,0.244,0.643
2023-03,744,669.167,31,30,0.899,0.968
2023-04,720,601.417,30,28,0.835,0.933
2023-05,744,591.833,31,28,0.795,0.903
2023-06,720,711.667,30,30,0.988,1.000
2023-07,744,735.583,31,31,0.989,1.000
2023-08,744,681.667,31,31,0.916,1.000
2023-09,720,705.333,30,30,0.980,1.000
2023-10,744,551.500,31,26,0.741,0.839


B. Whole month


,possible_hours,hours_with_data,possible_days,days_with_data,time_fraction,day_fraction
month,,,,,,
2023-02,672,81.917,28,9,0.122,0.321
2023-03,744,669.167,31,30,0.899,0.968
2023-04,720,601.417,30,28,0.835,0.933
2023-05,744,591.833,31,28,0.795,0.903
2023-06,720,711.667,30,30,0.988,1.000
2023-07,744,735.583,31,31,0.989,1.000
2023-08,744,681.667,31,31,0.916,1.000
2023-09,720,705.333,30,30,0.980,1.000
2023-10,744,551.500,31,26,0.741,0.839


## 3. Result in the layout of the DataAvailability tab

One row per method, one column per month, as a percentage (the lower block of the sheet uses percent with one decimal). The **time-based rows are the ones to use for the table**; the day-based rows are shown for reference. Only Feb 2023 and Feb 2024 differ between A and B.

In [5]:
# --- Sheet-style summaries (percent, months as columns) ------------------------
def sheet_style(table: pd.DataFrame) -> pd.DataFrame:
    """Percent of time and of days with data, one column per month."""
    out = (table[["time_fraction", "day_fraction"]].T * 100).round(1)
    out.index = ["FM-120: % of time with data", "FM-120: % of days with data"]
    out.columns = [m.strftime("%b %Y") for m in out.columns]
    return out


print("A. Campaign days only (Feb 2023 = Feb 15-28, Feb 2024 = Feb 1-14)")
display(sheet_style(avail))
print("B. Whole month (every month = all of its hours, incl. data after the campaign ended)")
display(sheet_style(avail_full))

A. Campaign days only (Feb 2023 = Feb 15-28, Feb 2024 = Feb 1-14)


,Feb 2023,Mar 2023,Apr 2023,May 2023,Jun 2023,Jul 2023,Aug 2023,Sep 2023,Oct 2023,Nov 2023,Dec 2023,Jan 2024,Feb 2024
FM-120: % of time with data,24.4,89.9,83.5,79.5,98.8,98.9,91.6,98.0,74.1,98.8,90.8,58.9,100.0
FM-120: % of days with data,64.3,96.8,93.3,90.3,100.0,100.0,100.0,100.0,83.9,100.0,93.5,80.6,100.0


B. Whole month (every month = all of its hours, incl. data after the campaign ended)


,Feb 2023,Mar 2023,Apr 2023,May 2023,Jun 2023,Jul 2023,Aug 2023,Sep 2023,Oct 2023,Nov 2023,Dec 2023,Jan 2024,Feb 2024
FM-120: % of time with data,12.2,89.9,83.5,79.5,98.8,98.9,91.6,98.0,74.1,98.8,90.8,58.9,50.7
FM-120: % of days with data,32.1,96.8,93.3,90.3,100.0,100.0,100.0,100.0,83.9,100.0,93.5,80.6,51.7


## 4. Periods when the instrument was not recording

A **gap** is a stretch of time with no 5-min sample. Each timestamp marks the *start* of its 5-min interval, so a sample at time *t* covers *t* to *t* + 5 min. Between two consecutive samples the gap therefore runs from (previous sample + 5 min) to (next sample), and adjacent samples (5 min apart) have no gap.

The stretches between the campaign start and the first sample, and between the last sample and the campaign end, count as gaps too.

FM-120 rows exist in clear air too (78 % of samples have N < 1 cm⁻³ and 88 % have LWC < 0.001 g m⁻³), and visibility at Mt. Soledad is distributed the same way inside and outside the gaps (visibility < 1 km 6.6 % vs 7.2 % of the time). So a gap means the instrument was not producing data, not that there was no cloud or haze.

In [6]:
# --- Find every gap -----------------------------------------------------------
# Start and end of every 5-min interval that has data.
interval_start = fm120.index.to_series()
interval_end = interval_start + SAMPLE_INTERVAL

# A gap begins where one interval ends and the next one does not start right away.
# Add the campaign start/end as boundaries, so that missing data at either end is counted.
gap_start = pd.concat([pd.Series([CAMPAIGN_START]), interval_end]).reset_index(drop=True)
gap_end = pd.concat([interval_start, pd.Series([CAMPAIGN_END])]).reset_index(drop=True)
gaps = pd.DataFrame({"start": gap_start, "end": gap_end})
gaps["duration"] = gaps["end"] - gaps["start"]
gaps = gaps[gaps["duration"] > pd.Timedelta(0)].reset_index(drop=True)

# Split into long gaps (1 day or longer, listed individually) and the rest (summarised).
LONG_GAP = pd.Timedelta(days=1)
long_gaps = gaps[gaps["duration"] >= LONG_GAP]
short_gaps = gaps[gaps["duration"] < LONG_GAP]

# Consistency check: time with data + time in gaps must equal the whole campaign window.
assert len(fm120) * SAMPLE_INTERVAL + gaps["duration"].sum() == CAMPAIGN_END - CAMPAIGN_START

print(f"{len(gaps)} gaps, {gaps['duration'].sum() / pd.Timedelta(hours=1):.1f} h in total")
print(f"{len(long_gaps)} of 1 day or longer, {len(short_gaps)} shorter than 1 day")
long_gaps

62 gaps, 1267.8 h in total
17 of 1 day or longer, 45 shorter than 1 day


,start,end,duration
0,2023-02-15 00:00:00,2023-02-16 16:40:00,1 days 16:40:00
2,2023-02-17 16:10:00,2023-02-20 10:40:00,2 days 18:30:00
3,2023-02-20 14:30:00,2023-02-22 12:25:00,1 days 21:55:00
4,2023-02-22 12:40:00,2023-02-24 15:15:00,2 days 02:35:00
11,2023-03-31 00:00:00,2023-04-01 00:00:00,1 days 00:00:00
15,2023-04-16 16:30:00,2023-04-19 00:00:00,2 days 07:30:00
22,2023-05-05 00:00:00,2023-05-08 01:55:00,3 days 01:55:00
25,2023-05-30 13:30:00,2023-05-31 17:55:00,1 days 04:25:00
32,2023-08-22 20:10:00,2023-08-23 20:50:00,1 days 00:40:00
38,2023-10-17 00:00:00,2023-10-19 02:35:00,2 days 02:35:00


## 5. Write the text report

Writes `FM120_data_availability.txt` next to this notebook: the monthly % of time with data, the gaps of one day or longer, statistics on the shorter gaps, and an hour-by-hour accounting of one month (`DETAIL_MONTH`, Feb 2023).

In [7]:
# --- Helpers for readable output ----------------------------------------------
def hours(td: pd.Timedelta) -> float:
    """Length of a time span in hours."""
    return td / pd.Timedelta(hours=1)


def stamp(t: pd.Timestamp) -> str:
    """Timestamp as 'YYYY-MM-DD HH:MM'."""
    return t.strftime("%Y-%m-%d %H:%M")


# --- Build the report line by line -------------------------------------------
total_time = CAMPAIGN_END - CAMPAIGN_START
lines = [
    "FM-120 Fog Droplet Monitor (Mt. Soledad), EPCAPE: data availability",
    "=" * 68,
    f"Data:     5-min averages, data/processed/FM120/Monthly FM-120 Files/ (UTC)",
    f"Campaign: {stamp(CAMPAIGN_START)} to {stamp(CAMPAIGN_END)} UTC (end exclusive)",
    f"Method:   % available = (number of 5-min samples x 5 min) / (possible time).",
    f"          Two versions of the possible time, differing only in the Februarys:",
    f"            A. campaign days only: Feb 2023 = Feb 15-28, Feb 2024 = Feb 1-14",
    f"            B. whole month: every month = all of its hours",
    f"          A counts only data inside the campaign window; B counts all data",
    f"          recorded in the month (adds 2024-02-15 00:00-17:00, after the campaign).",
    f"Made by:  EPCAPE/comparisons/data_availability/data_availability.ipynb",
    "",
    "1. Percent of time with data, by month",
    "-" * 40,
    f"{'':<10}{'A. Campaign days only':^30}  {'B. Whole month':^30}",
    f"{'Month':<10}{'Hours w/ data':>14}{'Possible':>10}{'%':>6}  {'Hours w/ data':>14}{'Possible':>10}{'%':>6}",
]
for m, a in avail.iterrows():
    b = avail_full.loc[m]
    lines.append(f"{m.strftime('%b %Y'):<10}"
                 f"{a['hours_with_data']:>14.1f}{a['possible_hours']:>10.0f}{100 * a['time_fraction']:>6.1f}  "
                 f"{b['hours_with_data']:>14.1f}{b['possible_hours']:>10.0f}{100 * b['time_fraction']:>6.1f}")
a_sum, b_sum = avail.sum(), avail_full.sum()
lines.append(f"{'All':<10}"
             f"{a_sum['hours_with_data']:>14.1f}{a_sum['possible_hours']:>10.0f}"
             f"{100 * a_sum['hours_with_data'] / a_sum['possible_hours']:>6.1f}  "
             f"{b_sum['hours_with_data']:>14.1f}{b_sum['possible_hours']:>10.0f}"
             f"{100 * b_sum['hours_with_data'] / b_sum['possible_hours']:>6.1f}")

lines += [
    "",
    "2. Periods with no data of 1 day or longer",
    "-" * 40,
    "Start = end of the last 5-min interval with data; End = start of the next one.",
    f"{'Start (UTC)':<19}{'End (UTC)':<19}{'Duration (days)':>16}",
]
for _, g in long_gaps.iterrows():
    lines.append(f"{stamp(g['start']):<19}{stamp(g['end']):<19}{hours(g['duration']) / 24:>16.2f}")
lines.append(f"Total: {len(long_gaps)} periods, {hours(long_gaps['duration'].sum()) / 24:.1f} days "
             f"({100 * long_gaps['duration'].sum() / total_time:.1f}% of the campaign)")

# Statistics on the shorter gaps, plus how many fall into a few duration classes.
d_h = short_gaps["duration"].map(hours)
classes = pd.cut(d_h, bins=[0, 1, 6, 12, 24], right=False,
                 labels=["< 1 h", "1-6 h", "6-12 h", "12-24 h"])
lines += [
    "",
    "3. Periods with no data shorter than 1 day",
    "-" * 40,
    f"Number of periods:  {len(short_gaps)}",
    f"Total time:         {d_h.sum():.1f} h ({100 * short_gaps['duration'].sum() / total_time:.1f}% of the campaign)",
    f"Median duration:    {d_h.median():.1f} h",
    f"Mean duration:      {d_h.mean():.1f} h",
    f"Shortest / longest: {d_h.min():.2f} h / {d_h.max():.1f} h",
    "Number by duration:",
]
for label, n in classes.value_counts(sort=False).items():
    lines.append(f"    {label:<8} {n:>3}")

# --- Detailed accounting of one month: every hour is either data or a gap -----
# Gaps that cross the month's edges (or the campaign window's) are clipped to it.
DETAIL_MONTH = pd.Period("2023-02", freq="M")
month_start = max(DETAIL_MONTH.start_time, CAMPAIGN_START)
month_end = min(DETAIL_MONTH.end_time.ceil("D"), CAMPAIGN_END)

month_gaps = gaps[(gaps["end"] > month_start) & (gaps["start"] < month_end)].copy()
month_gaps["start"] = month_gaps["start"].clip(lower=month_start)
month_gaps["end"] = month_gaps["end"].clip(upper=month_end)
month_gaps["duration"] = month_gaps["end"] - month_gaps["start"]

month_data = fm120[(fm120.index >= month_start) & (fm120.index < month_end)]
month_total = month_end - month_start
data_time = len(month_data) * SAMPLE_INTERVAL
gap_time = month_gaps["duration"].sum()
assert data_time + gap_time == month_total  # every hour of the month is accounted for

# Hours with data on each day of the month (0 on days with no samples).
days = pd.date_range(month_start, month_end, freq="D", inclusive="left")
data_hours_per_day = (month_data.groupby(month_data.index.normalize()).size()
                      .reindex(days, fill_value=0) * hours(SAMPLE_INTERVAL))

lines += [
    "",
    f"4. {DETAIL_MONTH.strftime('%b %Y')} in detail",
    "-" * 40,
    f"Window: {stamp(month_start)} to {stamp(month_end)} UTC = {hours(month_total) / 24:.0f} days "
    f"({hours(month_total):.0f} h)",
    f"With data: {len(month_data)} samples x 5 min = {hours(data_time):.1f} h "
    f"({hours(data_time) / 24:.2f} days, {100 * data_time / month_total:.1f}%)",
    f"Missing:   {hours(gap_time):.1f} h ({hours(gap_time) / 24:.2f} days, "
    f"{100 * gap_time / month_total:.1f}%) in {len(month_gaps)} periods:",
    f"    {'Start (UTC)':<19}{'End (UTC)':<19}{'Hours':>7}{'Days':>7}",
]
for _, g in month_gaps.iterrows():
    lines.append(f"    {stamp(g['start']):<19}{stamp(g['end']):<19}"
                 f"{hours(g['duration']):>7.1f}{hours(g['duration']) / 24:>7.2f}")
lines.append("Hours with data on each day (of 24):")
for day, h in data_hours_per_day.items():
    lines.append(f"    {day.strftime('%Y-%m-%d')}  {h:5.2f}")

lines += [
    "",
    "Note: the FM-120 records clear air as well (most samples have N < 1 cm-3), so",
    "a period with no data means the instrument was not recording, not that there",
    "was no cloud or haze.",
]

# --- Write and show the report ------------------------------------------------
REPORT = Path.cwd() / "FM120_data_availability.txt"
REPORT.write_text("\n".join(lines) + "\n")
print(REPORT.read_text())

FM-120 Fog Droplet Monitor (Mt. Soledad), EPCAPE: data availability
Data:     5-min averages, data/processed/FM120/Monthly FM-120 Files/ (UTC)
Campaign: 2023-02-15 00:00 to 2024-02-15 00:00 UTC (end exclusive)
Method:   % available = (number of 5-min samples x 5 min) / (possible time).
          Two versions of the possible time, differing only in the Februarys:
            A. campaign days only: Feb 2023 = Feb 15-28, Feb 2024 = Feb 1-14
            B. whole month: every month = all of its hours
          A counts only data inside the campaign window; B counts all data
          recorded in the month (adds 2024-02-15 00:00-17:00, after the campaign).
Made by:  EPCAPE/comparisons/data_availability/data_availability.ipynb

1. Percent of time with data, by month
----------------------------------------
              A. Campaign days only               B. Whole month        
Month      Hours w/ data  Possible     %   Hours w/ data  Possible     %
Feb 2023            81.9       336  24.4   

# Part 2: JPL radars at the Scripps Pier (Matt Lebsock), sheet row 49

**What we know about these data** (email thread Lynn Russell – Matt Lebsock, 2026-10-12):

* Matt: data are available **March 23 – April 27** (2023), "during cloudy conditions (LCF > 0, 8 AM – 6 PM LST)". He reported no flagged times.
* Lynn: "compute %time using daytime only (8-6, or SZA threshold?) using ARM data for detected clouds. No need to 'retrieve' low cloud fraction from his radar data."

So the radar files themselves are not needed: the radars are taken to have data whenever it was **daytime** and a **low cloud** was overhead during the deployment.

**Definition used here.** For each month,

$$\text{availability} = \frac{\text{time inside the deployment window that is daytime AND has a low cloud over the Pier}}{\text{all hours in the calendar month}}$$

The denominator is the whole month, as for the FM-120 whole-month table (B) and the sheet header "Data Availability (% of month)".

* **Low cloud** (the same test as for the FM-120 cloudy periods, using the Pier instrument): the ARM Vaisala ceilometer at the Pier (`epcceilM1.b1`, one sample every 16 s) reports a cloud base (`detection_status` 1–3) whose height passes QC (`qc_first_cbh` = 0) and is ≤ 3500 m. This is `ceilometer_low_cloud_state()` in `comparisons/seasonal_averages/quantities.py`.
* **Daytime**, two ways, as requested:
  1. **Clock hours:** 08:00–18:00 LST.
  2. **Sun angle:** solar zenith angle (SZA) < 80°, i.e. the sun at least 10° above the horizon.

**Choices to check:**

* The request said "SZA > 80°", but SZA > 80° is the sun within 10° of the horizon or below it (twilight and night). The daytime test is SZA **< 80°**, which is what is used here.
* "LST" is read literally as local *standard* time, PST = UTC − 8 h. During Mar–Apr 2023 the local clock was on daylight time (PDT = UTC − 7 h), so if Matt meant clock time, the window is 1 h earlier in UTC. `UTC_OFFSET_H` below changes this.
* The deployment window is taken as 2023-03-23 00:00 to 2023-04-28 00:00 LST, i.e. both end dates **included**.
* Samples when the ceilometer could not tell (`detection_status` 4: fog or heavy rain hides any base; or a base failing QC) would **not** count as cloudy. In practice there are none: status 4 never occurs in the campaign's Pier ceilometer record, and no detected base in the radar window fails QC (`unknown_h` = 0 in Section 7).

In [8]:
# --- Settings for Part 2 -------------------------------------------------------
import sys

import numpy as np
import xarray as xr

# Make the EPCAPE package importable (its PARENT folder goes on sys.path), to reuse the
# ceilometer cloud test and the solar zenith angle used elsewhere in the EPCAPE analysis.
if str(EPCAPE_DIR.parent) not in sys.path:
    sys.path.insert(0, str(EPCAPE_DIR.parent))
from EPCAPE.comparisons.seasonal_averages import quantities as Q

# Pier ceilometer (ARM epcceilM1.b1), combined over the campaign by download_data/combine_product.py.
CEIL_FILE = EPCAPE_DIR / "data" / "processed" / "cbh_ceil_M1_20230215_20240214.nc"

# Local standard time = UTC + UTC_OFFSET_H (PST = UTC - 8 h). Use -7 for clock time (PDT) in Mar-Apr.
UTC_OFFSET_H = -8

# Radar deployment window in LOCAL time, from Matt's email: Mar 23 - Apr 27 2023, both days included.
RADAR_START_LST = pd.Timestamp("2023-03-23 00:00")
RADAR_END_LST = pd.Timestamp("2023-04-28 00:00")   # exclusive

# Daytime definitions.
DAY_START_HOUR_LST, DAY_END_HOUR_LST = 8, 18   # (1) 08:00 <= local time < 18:00
MAX_SZA_DEG = 80.0                             # (2) solar zenith angle < 80 deg

# Low cloud: ceilometer cloud base at or below this height (m above the instrument), as for the FM-120.
MAX_CBH_M = 3500.0

## 6. Classify every ceilometer sample: low cloud, daytime, inside the window

Each 16-s ceilometer sample gets three yes/no labels. The sample then stands for 16 s of time, so the time that passes all three tests is (number of samples passing) × 16 s, the same counting used for the FM-120 (number of 5-min samples × 5 min). The ceilometer record is nearly continuous in Mar–Apr 2023: its longest gap there is 7.5 min (2023-04-20).

In [9]:
# --- Load the Pier ceilometer -------------------------------------------------
ceil = xr.open_dataset(CEIL_FILE)
pier_lat_deg, pier_lon_deg = float(ceil["lat"]), float(ceil["lon"])   # 32.87 N, -117.26 E

# Sample spacing: the ceilometer reports every 16 s; each sample stands for this much time.
times_utc = pd.DatetimeIndex(ceil["time"].values)
SAMPLE_DT = pd.Timedelta(seconds=float(np.median(np.diff(times_utc.asi8)) / 1e9))
print("Ceilometer sample spacing:", SAMPLE_DT)

# --- Label 1: low cloud overhead ----------------------------------------------
# 1 = cloud base detected (status 1-3), QC = 0 and base <= MAX_CBH_M
# 0 = clear (status 0 or 5) or only cloud above MAX_CBH_M
# NaN = unknown (status 4: fog/heavy rain hides the base, or the base fails QC)
cloud_state = Q.ceilometer_low_cloud_state(ceil, max_cbh_m=MAX_CBH_M)

samples = pd.DataFrame({"cloud_state": cloud_state.values}, index=times_utc)
samples["low_cloud"] = samples["cloud_state"] == Q.LOW_CLOUD
samples["unknown"] = samples["cloud_state"].isna()

# --- Local standard time for each sample ----------------------------------------
samples["time_lst"] = samples.index + pd.Timedelta(hours=UTC_OFFSET_H)

# --- Label 2: inside the radar deployment window (local time) ------------------
samples["in_window"] = (samples["time_lst"] >= RADAR_START_LST) & (samples["time_lst"] < RADAR_END_LST)

# --- Label 3a: daytime by clock hours (08:00-18:00 LST) ------------------------
local_hour = samples["time_lst"].dt.hour
samples["day_clock"] = (local_hour >= DAY_START_HOUR_LST) & (local_hour < DAY_END_HOUR_LST)

# --- Label 3b: daytime by sun angle (SZA < 80 deg) -----------------------------
# Q.solar_zenith_deg: NOAA GML "General Solar Position Calculations" (Spencer 1971 Fourier series
# for declination and equation of time), evaluated at each sample's UTC time at the Pier.
samples["sza_deg"] = Q.solar_zenith_deg(samples.index, pier_lat_deg, pier_lon_deg)
samples["day_sza"] = samples["sza_deg"] < MAX_SZA_DEG

# Month of each sample, by local time (so a local day is never split between months).
samples["month"] = samples["time_lst"].dt.to_period("M")

window = samples[samples["in_window"]]
print(f"Samples inside the radar window: {len(window)} "
      f"({len(window) * SAMPLE_DT / pd.Timedelta(hours=1):.1f} h of ceilometer record; "
      f"window length {(RADAR_END_LST - RADAR_START_LST) / pd.Timedelta(hours=1):.0f} h)")

Ceilometer sample spacing: 0 days 00:00:16


Samples inside the radar window: 194342 (863.7 h of ceilometer record; window length 864 h)


## 7. Hours available per month, for both daytime definitions

For each month and daytime definition the table lists:

* `daytime_h`: daytime hours inside the deployment window (the radar's maximum possible time). These come from the ceilometer timestamps, so they are slightly below the exact clock value wherever the ceilometer has short gaps.
* `unknown_h`: daytime hours when the ceilometer could not decide (status 4 or a base failing QC).
* `low_cloud_h`: daytime hours with a low cloud overhead = **radar data available**.
* `pct_of_month`: `low_cloud_h` / all hours in the month × 100 (the number for the sheet).
* `pct_of_daytime`: `low_cloud_h` / `daytime_h` × 100, for reference: how often it was cloudy during the radar's daytime window.

In [10]:
# --- Hours in every calendar month of the campaign (the denominator) -----------
months = pd.period_range(FULL_MONTHS_START, FULL_MONTHS_END - pd.Timedelta(days=1), freq="M")
hours_in_month = pd.Series({m: m.days_in_month * 24 for m in months})


def to_hours(n_samples: pd.Series) -> pd.Series:
    """Number of ceilometer samples -> hours (each sample stands for SAMPLE_DT)."""
    return n_samples * SAMPLE_DT / pd.Timedelta(hours=1)


def radar_availability(day_column: str) -> pd.DataFrame:
    """Monthly radar availability with daytime defined by the boolean column `day_column`."""
    day = window[window[day_column]]                  # daytime samples inside the deployment window
    by_month = day.groupby("month")
    table = pd.DataFrame({
        "hours_in_month": hours_in_month,
        "daytime_h": to_hours(by_month.size()),
        "unknown_h": to_hours(by_month["unknown"].sum()),
        "low_cloud_h": to_hours(by_month["low_cloud"].sum()),
    }).fillna(0.0)
    table["pct_of_month"] = 100 * table["low_cloud_h"] / table["hours_in_month"]
    table["pct_of_daytime"] = 100 * table["low_cloud_h"] / table["daytime_h"].where(table["daytime_h"] > 0)
    table.index.name = "month"
    return table


radar_clock = radar_availability("day_clock")
radar_sza = radar_availability("day_sza")

print("Daytime = 08:00-18:00 LST")
display(radar_clock.loc["2023-03":"2023-04"].round(1))
print(f"Daytime = SZA < {MAX_SZA_DEG:g} deg")
display(radar_sza.loc["2023-03":"2023-04"].round(1))
print("All other months: 0 h (outside the deployment window).")

Daytime = 08:00-18:00 LST


,hours_in_month,daytime_h,unknown_h,low_cloud_h,pct_of_month,pct_of_daytime
month,,,,,,
2023-03,744,90.0,0.0,24.0,3.2,26.7
2023-04,720,269.9,0.0,128.1,17.8,47.5


Daytime = SZA < 80 deg


,hours_in_month,daytime_h,unknown_h,low_cloud_h,pct_of_month,pct_of_daytime
month,,,,,,
2023-03,744,95.5,0.0,29.0,3.9,30.3
2023-04,720,302.1,0.0,146.3,20.3,48.4


All other months: 0 h (outside the deployment window).


## 8. Result in the layout of the DataAvailability tab (row 49)

Percent of the whole month, one column per month. The sheet currently holds 10.8 % (Mar) and 37.5 % (Apr). These equal 80 h / 744 h and 270 h / 720 h: daytime hours (10 h a day) with **no** cloud test, with March apparently counting 8 days rather than the 9 of Mar 23–31.

**Effect of the time zone.** With clock time (PDT, `UTC_OFFSET_H = -7`) instead of PST, the 08:00–18:00 row changes from 3.2 % to 3.6 % (Mar) and from 17.8 % to 17.9 % (Apr). The SZA row does not depend on the time zone except through which local day, and so which month, a sample falls in.

In [11]:
# --- Sheet-style row (percent of the whole month) ------------------------------
row49 = pd.DataFrame({
    "JPL radars: daytime 08-18 LST, low cloud": radar_clock["pct_of_month"],
    f"JPL radars: SZA < {MAX_SZA_DEG:g} deg, low cloud": radar_sza["pct_of_month"],
}).T.round(1)
row49.columns = [m.strftime("%b %Y") for m in row49.columns]
row49

,Feb 2023,Mar 2023,Apr 2023,May 2023,Jun 2023,Jul 2023,Aug 2023,Sep 2023,Oct 2023,Nov 2023,Dec 2023,Jan 2024,Feb 2024
"JPL radars: daytime 08-18 LST, low cloud",0.0,3.2,17.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"JPL radars: SZA < 80 deg, low cloud",0.0,3.9,20.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
